<a href="https://colab.research.google.com/github/aps817/Greatlearning/blob/main/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -qU \
    "chromadb==1.5.9" \
    "langchain-community==0.4.1" \
    "langchain-chroma==1.1.0" \
    "langchain-openai==1.2.1" \
    "langchain-text-splitters==1.1.2" \
    "pandas>=2.2.0" \
    "numpy>=1.26.0" \
    "scikit-learn>=1.4.0" \
    "python-dotenv>=1.0.1" \
    "tqdm>=4.66.0" \
    "pypdf>=5.0.0" \
    "deepeval"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 964.1/964.1 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0

*Prompt:*

I want to build and evaluate a Retrieval-Augmented Generation (RAG) system for health insurance policy intelligence. Help me install and import the necessary Python libraries to:

Read and manipulate structured data such as JSON and tabular datasets
Work with environment variables, file paths, and runtime utilities
Load and process health insurance policy PDF documents
Split long policy documents into smaller semantic chunks
Generate vector embeddings using OpenAI embedding models
Store and retrieve embeddings using Chroma vector database
Build RAG pipelines using LangChain components and chat models
Optimize prompts using DeepEval and the GEPA prompt optimization algorithm
Evaluate generated answers using relevance, faithfulness, and completeness metrics
Create and manage benchmark datasets for prompt optimization and RAG evaluation



In [2]:
# Importing the necessary libraries

# Standard Python libraries for file handling, text processing, JSON handling,
# randomness, timing, and basic utilities.
import os
import re
import json
import random
import time
from pathlib import Path
from collections import Counter
import pandas as pd

# DeepEval libraries for prompt optimization and evaluation.
from deepeval.prompt import Prompt
from deepeval.dataset import Golden
from deepeval.metrics import GEval
from deepeval.optimizer import PromptOptimizer
from deepeval.optimizer.algorithms import GEPA
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.models import GPTModel
from deepeval.optimizer.policies import TieBreaker

# LangChain libraries for document loading, text splitting, embeddings,
# vector storage, and LLM interaction.
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

## Define Input Data Paths

In [3]:
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

print(f"Data directory created at: {DATA_DIR.resolve()}")

Data directory created at: /content/data


## Upload Health Insurance Policy PDF Documents

Please upload your health insurance policy PDF documents to the newly created `data` directory (e.g., `/content/data/policy_document.pdf`). Once the files are uploaded, we can proceed to load and process them.

In [4]:
# Define the local paths for all input datasets.
# Place the required files inside the ./data directory.

DATA_DIR = Path("data")

POLICY_BROCHURE_PDF = DATA_DIR / "Policy_Brocher.pdf"
POLICY_TERMS_PDF = DATA_DIR / "Health_Insurance_Policy_TandC.pdf"
CLAIM_EXCLUSIONS_PDF = DATA_DIR / "List_of_Standard_Claim_Exclusions.pdf"
CLAIM_GUIDE_PDF = DATA_DIR / "Health_Insurance_Claim_Guide.pdf"


BENCHMARK_CSV = DATA_DIR / "golden_benchmark_dataset.csv"

In [2]:
from pathlib import Path

# Define local directories for:
# 1. Chroma vector database persistence
# 2. Saved DeepEval optimized prompts, evaluation outputs, and RAG tuning artifacts

CHROMA_DIR = Path("chroma_db")
ARTIFACT_DIR = Path("artifacts")

# Create the directories automatically if they do not already exist.
CHROMA_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)

In [4]:
import os

# Request OpenAI credentials if they are not already available.
# These credentials are used for:
# - embedding generation
# - answer generation
# - DeepEval optimization and evaluation
if not os.getenv("OPENAI_API_KEY"):
    import getpass
    os.environ["OPENAI_API_KEY"] = "gl-U2FsdGVkX19vUNc5KmspIA4doF+7pd+2rvV8etMJgVrbGAJJK2EjYrY0MtEVrGV8" # getpass.getpass("Enter OPENAI_API_KEY: ")

if not os.getenv("OPENAI_BASE_URL"):
    import getpass
    os.environ["OPENAI_BASE_URL"] = "https://aibe.mygreatlearning.com/openai/v1" #getpass.getpass("Enter OPENAI_BASE_URL: ")

In [5]:
# Define the models used throughout the notebook.

# Embedding model:
# Converts health insurance policy text into vector embeddings.
EMBEDDING_MODEL = "text-embedding-3-small"

# Answer generation model:
# Used for baseline RAG generation, DeepEval prompt optimization,
# RAG configuration tuning, and final policy intelligence inference.
ANSWER_MODEL = "gpt-4o-mini"
EVAL_MODEL = "gpt-4o-mini"

In [7]:
from pathlib import Path

# Define the local paths for all input datasets.
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Define local directories for Chroma vector database persistence and artifacts
CHROMA_DIR = Path("chroma_db")
ARTIFACT_DIR = Path("artifacts")

# Create the directories automatically if they do not already exist.
CHROMA_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)

# Display the active working directories.
print("Data directory:", DATA_DIR.resolve())
print("Chroma directory:", CHROMA_DIR.resolve())
print("Artifacts directory:", ARTIFACT_DIR.resolve())

Data directory: /content/data
Chroma directory: /content/chroma_db
Artifacts directory: /content/artifacts


I want to load multiple health insurance policy PDF documents into LangChain so that each page can be used as a retrievable document. Help me create functions to:

Check whether the required PDF file exists locally before loading it
Load each PDF using PyPDFLoader
Convert every page into a LangChain document
Add source-level metadata such as file name and file type for traceability
Load the policy brochure, policy terms and conditions, claim exclusions, and claim guide as separate document sets

In [8]:
def ensure_file_exists(path: Path) -> None:
    """
    Check whether the required input file exists locally.

    Raises a clear error if the file is missing so the pipeline
    stops before document loading begins.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}. Put it under {DATA_DIR.resolve()} and run again."
        )

In [9]:
def load_pdf_documents(path: Path):
    """
    Load a PDF file using LangChain's PyPDFLoader.

    Each page becomes one LangChain Document, and the source file
    information is preserved for traceability.
    """
    ensure_file_exists(path)

    loader = PyPDFLoader(str(path))
    docs = loader.load()

    # Add dataset-level metadata to every document
    # so the source type remains identifiable later.
    for doc in docs:
        doc.metadata["source_file"] = path.name
        doc.metadata["source_type"] = "pdf"

    return docs

In [16]:
from pathlib import Path

# Ensure langchain-community is installed if not already available.
try:
    from langchain_community.document_loaders import PyPDFLoader
except ImportError:
    print("Installing langchain-community...")
    %pip install -q "langchain-community==0.4.1"
    from langchain_community.document_loaders import PyPDFLoader # Re-import after installation

# Ensure pypdf is installed as it's a dependency for PyPDFLoader
try:
    import pypdf
except ImportError:
    print("Installing pypdf...")
    %pip install -q pypdf
    import pypdf # Re-import after installation

# Define the local paths for all input datasets.
DATA_DIR = Path("data")

POLICY_BROCHURE_PDF = DATA_DIR / "Policy_Brocher.pdf"
POLICY_TERMS_PDF = DATA_DIR / "Health_Insurance_Policy_TandC.pdf"
CLAIM_EXCLUSIONS_PDF = DATA_DIR / "List_of_Standard_Claim_Exclusions.pdf"
CLAIM_GUIDE_PDF = DATA_DIR / "Health_Insurance_Claim_Guide.pdf"

def ensure_file_exists(path: Path) -> None:
    """
    Check whether the required input file exists locally.

    Raises a clear error if the file is missing so the pipeline
    stops before document loading begins.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}. Put it under {DATA_DIR.resolve()} and run again."
        )

def load_pdf_documents(path: Path):
    """
    Load a PDF file using LangChain's PyPDFLoader.

    Each page becomes one LangChain Document, and the source file
    information is preserved for traceability.
    """
    ensure_file_exists(path)

    loader = PyPDFLoader(str(path))
    docs = loader.load()

    # Add dataset-level metadata to every document
    # so the source type remains identifiable later.
    for doc in docs:
        doc.metadata["source_file"] = path.name
        doc.metadata["source_type"] = "pdf"

    return docs

# Load policy brochure pages as LangChain Documents.
brochure_docs = load_pdf_documents(POLICY_BROCHURE_PDF)

# Load policy terms and conditions pages as LangChain Documents.
terms_docs = load_pdf_documents(POLICY_TERMS_PDF)

# Load standard claim exclusions pages as LangChain Documents.
exclusions_docs = load_pdf_documents(CLAIM_EXCLUSIONS_PDF)

# Load claim guide pages as LangChain Documents.
claim_guide_docs = load_pdf_documents(CLAIM_GUIDE_PDF)

# Display basic verification information
# to confirm document loading worked correctly.
print(f"Loaded brochure documents:   {len(brochure_docs)}")
print(f"Loaded terms documents:      {len(terms_docs)}")
print(f"Loaded exclusions documents: {len(exclusions_docs)}")
print(f"Loaded claim guide docs:     {len(claim_guide_docs)}")

print("Brochure doc example:")
print(brochure_docs[0].page_content[:700])

print("\nMetadata:", brochure_docs[0].metadata)

Installing pypdf...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 3.6 MB/s eta 0:00:00
Loaded brochure documents:   31
Loaded terms documents:      28
Loaded exclusions documents: 8
Loaded claim guide docs:     3
Brochure doc example:
Family Health Insurance Plans 
A complete guide to protecting your loved ones with a single, shared health insurance plan — covering 
benefits, features, eligibility, claims, exclusions, and how to choose the right cover for your household. 
 
 
What Is a Family Health Insurance Plan? 
A family health insurance plan, often called a family floater, allows you to cover several members of your 
household — typically up to two adults and two children — under one single policy. You pay a single 
premium, and everyone insured shares one common sum insured. Instead of juggling multiple individual 
policies, your family gets one consolidated cover that any insured member can draw on during a medical

Metadata: {'producer': 'www.ilovepdf.com', 'creato

*Prompt:*

I want to prepare the loaded PDF pages for semantic retrieval in a RAG pipeline. Help me write code to:

Combine all loaded policy documents into one dataset
Split the documents into smaller text chunks using RecursiveCharacterTextSplitter
Use a chunk size and overlap that preserve context across adjacent chunks
Create chunked documents that are suitable for embeddings and vector search
Print verification details such as total source documents, total chunks, and sample chunk metadata

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Combine all source documents before chunking.
all_source_docs = brochure_docs + terms_docs + exclusions_docs + claim_guide_docs

# Configure the text splitter used for chunking.
#   chunk_size: Maximum number of characters per chunk.
#   chunk_overlap: Overlapping text between chunks to preserve context continuity.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
)

# Split the PDF pages into smaller chunks for embedding generation and semantic retrieval.
policy_chunks = splitter.split_documents(all_source_docs)

# Display verification information to confirm chunking worked correctly.
print(f"Loaded source documents: {len(all_source_docs)}")
print(f"Split policy chunks:     {len(policy_chunks)}")

print("Policy chunk example:")
print(policy_chunks[0].page_content[:700])

print("\nMetadata:", policy_chunks[0].metadata)

Loaded source documents: 70
Split policy chunks:     195
Policy chunk example:
Family Health Insurance Plans 
A complete guide to protecting your loved ones with a single, shared health insurance plan — covering 
benefits, features, eligibility, claims, exclusions, and how to choose the right cover for your household. 
 
 
What Is a Family Health Insurance Plan? 
A family health insurance plan, often called a family floater, allows you to cover several members of your 
household — typically up to two adults and two children — under one single policy. You pay a single 
premium, and everyone insured shares one common sum insured. Instead of juggling multiple individual 
policies, your family gets one consolidated cover that any insured member can draw on during a medical

Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-07T09:55:46+00:00', 'moddate': '2026-05-07T09:55:47+00:00', 'source': 'data/Policy_Brocher.pdf', 'total_pages': 31, 

*Prompt:*

I want to store the chunked health insurance policy documents in a local vector database for retrieval. Help me write code to:

Generate embeddings for the chunked documents using OpenAI embeddings
Create a local Chroma vector store
Persist the vector database in a local directory
Store all chunked policy documents in one searchable collection
Print confirmation that the vector store is ready and the documents have been indexed

In [25]:
# Create embeddings and index everything into Chroma.

# Ensure langchain-openai is installed if not already available.
try:
    from langchain_openai import OpenAIEmbeddings
except ImportError:
    print("Installing langchain-openai...")
    %pip install -q "langchain-openai==1.2.1"
    from langchain_openai import OpenAIEmbeddings # Re-import after installation

# Ensure langchain-chroma is installed if not already available.
try:
    from langchain_chroma import Chroma
except ImportError:
    print("Attempting to resolve chromadb/opentelemetry conflicts and install langchain-chroma...")
    # The error indicates an opentelemetry dependency conflict with chromadb.
    # We will attempt a more aggressive uninstall of all opentelemetry packages,
    # chromadb, and langchain-chroma to ensure a clean slate.

    print("Uninstalling all opentelemetry packages, chromadb, and langchain-chroma (if present)...")
    %pip uninstall -y\
        opentelemetry-sdk\
        opentelemetry-api\
        opentelemetry-exporter-otlp\
        opentelemetry-proto\
        opentelemetry-exporter-otlp-proto-common\
        opentelemetry-exporter-otlp-proto-grpc\
        opentelemetry-exporter-otlp-proto-http\
        opentelemetry-instrumentation\
        opentelemetry-semantic-conventions\
        chromadb\
        langchain-chroma

    # Explicitly install compatible opentelemetry versions that include the required symbol.
    # These versions are compatible with chromadb==1.5.9's requirements (>=1.13.0, <2.0.0)
    # and also include 'OTEL_PYTHON_SDK_INTERNAL_METRICS_ENABLED' (introduced in >=1.17.0).
    print("Installing specific opentelemetry versions...")
    %pip install -q \
        "opentelemetry-sdk==1.22.0" \
        "opentelemetry-api==1.22.0" \
        "opentelemetry-exporter-otlp==1.22.0" \
        "opentelemetry-proto==1.22.0" \
        "opentelemetry-exporter-otlp-proto-common==1.22.0" \
        "opentelemetry-exporter-otlp-proto-grpc==1.22.0"

    # Then reinstall chromadb and langchain-chroma. Pip should ideally
    # respect the already installed opentelemetry packages if they satisfy dependencies.
    print("Reinstalling chromadb and langchain-chroma...")
    %pip install -q "chromadb==1.5.9" "langchain-chroma==1.1.0"

    from langchain_chroma import Chroma # Re-import after installation

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

all_documents = policy_chunks

# Chroma persists locally inside CHROMA_DIR.
vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings,
    collection_name="health_policy_intelligence",
    persist_directory=str(CHROMA_DIR),
)

print(f"Indexed documents: {len(all_documents)}")
print("Vector store is ready.")

Attempting to resolve chromadb/opentelemetry conflicts and install langchain-chroma...
Uninstalling all opentelemetry packages, chromadb, and langchain-chroma (if present)...
Found existing installation: opentelemetry-sdk 1.42.1
Uninstalling opentelemetry-sdk-1.42.1:
  Successfully uninstalled opentelemetry-sdk-1.42.1
Found existing installation: opentelemetry-api 1.42.1
Uninstalling opentelemetry-api-1.42.1:
  Successfully uninstalled opentelemetry-api-1.42.1
Found existing installation: opentelemetry-proto 1.42.1
Uninstalling opentelemetry-proto-1.42.1:
  Successfully uninstalled opentelemetry-proto-1.42.1
Found existing installation: opentelemetry-exporter-otlp-proto-common 1.42.1
Uninstalling opentelemetry-exporter-otlp-proto-common-1.42.1:
  Successfully uninstalled opentelemetry-exporter-otlp-proto-common-1.42.1
Found existing installation: opentelemetry-exporter-otlp-proto-grpc 1.42.1
Uninstalling opentelemetry-exporter-otlp-proto-grpc-1.42.1:
  Successfully uninstalled opentele

I want to build a simple baseline RAG pipeline for answering health insurance policy questions. Help me write code to:

Initialize a chat model for answer generation
Retrieve the most relevant chunks from the Chroma vector store for a user question
Format retrieved documents into a compact context block with metadata
Create a very simple baseline system prompt and user prompt
Keep the question and retrieved context dynamic during runtime
Generate an answer using only the retrieved policy context
Return the answer, retrieved documents, and formatted context from a reusable function
Run one sample policy question to verify the full end-to-end workflow

In [28]:
from langchain_openai import ChatOpenAI

# Initialize the answer-generation language model.
# temperature=0 keeps responses more deterministic and consistent.
llm = ChatOpenAI(
    model=ANSWER_MODEL,
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

In [36]:
from typing import List, Tuple
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage

def build_and_run_rag_pipeline(
    question: str,
    llm: ChatOpenAI,
    vectorstore: Chroma,
    k_retrieved_documents: int = 3,
) -> Tuple[str, List[Document], str]:
    """
    Builds and runs a simple RAG pipeline to answer a question.

    Args:
        question (str): The user's question.
        llm (ChatOpenAI): The initialized chat model.
        vectorstore (Chroma): The initialized Chroma vector store.
        k_retrieved_documents (int): The number of relevant documents to retrieve.

    Returns:
        Tuple[str, List[Document], str]: A tuple containing the generated answer,
                                         the list of retrieved documents, and
                                         the formatted context string.
    """

    # 1. Retrieve the most relevant chunks from the Chroma vector store
    retrieved_docs = vectorstore.similarity_search(query=question, k=k_retrieved_documents)

    # 2. Format retrieved documents into a compact context block with metadata
    formatted_context = ""
    for i, doc in enumerate(retrieved_docs):
        formatted_context += f"-- Document {i+1} (Source: {doc.metadata.get('source_file', 'N/A')}, Page: {doc.metadata.get('page_label', 'N/A')}) --\n"
        formatted_context += doc.page_content + "\n\n"

    # 3. Create a very simple baseline system prompt and user prompt
    system_prompt_template = (
        "You are an AI assistant specialized in health insurance policies. "
        "Answer the user's question only based on the provided policy context. "
        "If the answer is not found in the context, state that you don't have enough information. "
        "Do not make up any information."
    )

    user_prompt_template = (
        f"Context:\n{formatted_context}"
        f"Question: {question}"
        f"Answer:"
    )

    messages = [
        SystemMessage(content=system_prompt_template),
        HumanMessage(content=user_prompt_template),
    ]

    # 4. Generate an answer using only the retrieved policy context
    response = llm.invoke(messages)
    answer = response.content

    return answer, retrieved_docs, formatted_context

# Run one sample policy question to verify the full end-to-end workflow
sample_question = "What are the eligibility criteria for a family health insurance plan?"

print(f"User Question: {sample_question}\n")

r_answer, r_docs, r_context = build_and_run_rag_pipeline(
    question=sample_question,
    llm=llm,
    vectorstore=vectorstore,
    k_retrieved_documents=3,
)

print("--- Generated Answer ---")
print(r_answer)

print("\n--- Retrieved Context ---")
print(r_context)

print("\n--- Retrieved Documents (Metadata Only) ---")
for i, doc in enumerate(r_docs):
    print(f"Document {i+1} Metadata: {doc.metadata}")

User Question: What are the eligibility criteria for a family health insurance plan?

--- Generated Answer ---
The eligibility criteria for a family health insurance plan are as follows:

- Minimum Entry Age: 91 days for floater policy, provided at least one insured person is aged 18 or above. For individual policy, the minimum entry age is 18 years.
- Maximum Entry Age: Lifelong renewability is generally available.
- Premium Payable on Renewal: May change on renewal, subject to prior approval from the relevant insurance regulator.
- Waiting Periods: Initial waiting period is 30 days; for listed specific ailments, it is 24 months; for pre-existing conditions, it is 36 months.
- Grace Period: Typically 30 days after expiry to renew without losing accumulated benefits.

--- Retrieved Context ---
-- Document 1 (Source: Policy_Brocher.pdf, Page: 6) --
Eligibility Criteria for Family Health Insurance 
Cashless family health insurance plans are available to applicants who meet the eligibilit

In [30]:
# Initialize the answer-generation language model.
# temperature=0 keeps responses more deterministic and consistent.
llm = ChatOpenAI(
    model=ANSWER_MODEL,
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

In [37]:
import json

def retrieve_docs(question: str, k: int = 4):
    """
    Retrieve the most semantically relevant document chunks
    from the Chroma vector database.
    """
    return vectorstore.similarity_search(question, k=k)


def format_docs(docs, max_chars: int = 1200) -> str:
    """
    Convert retrieved documents into a compact context block.

    The formatted context is later inserted into the prompt
    before sending it to the language model.
    """
    blocks = []

    for i, doc in enumerate(docs, start=1):
        # Clean and shorten the retrieved text chunk.
        snippet = doc.page_content.strip().replace("\n", " ")

        if len(snippet) > max_chars:
            snippet = snippet[:max_chars] + "..."

        # Attach metadata so retrieved sources remain visible.
        blocks.append(
            f"[Source {i}] {snippet}\n"
            f"Metadata: {json.dumps(doc.metadata, ensure_ascii=False)}"
        )

    return "\n\n".join(blocks)

In [32]:
# -------------------------------------------------------------------
# Baseline RAG Prompt Templates
# -------------------------------------------------------------------

BASELINE_SYSTEM_PROMPT = """
You are a health insurance assistant.
Answer the question using the context.
""".strip()

In [33]:
BASELINE_USER_PROMPT = """
Broker Question:
{question}

Retrieved Context:
{context}
""".strip()

Note: The following part of the code in the BASELINE_USER_PROMPT ensures that the question and context values are dynamically fetched during runtime.

Broker Question:
{question}
Retrieved Context:
{context}
Please ensure that this part remains UNCHANGED.

In [39]:
import json
from langchain_core.messages import SystemMessage, HumanMessage

def baseline_answer(question: str, k: int = 4) -> dict:
    """
    Generate an answer using the baseline RAG workflow.

    Steps:
    1. Retrieve relevant policy context
    2. Build the grounded prompt
    3. Send the prompt to the LLM
    4. Return the generated answer and retrieved sources
    """

    # Retrieve relevant chunks from Chroma
    docs = retrieve_docs(question, k=k)

    # Convert retrieved documents into prompt-ready context
    context = format_docs(docs)

    # Inject the runtime values into the prompt template
    user_prompt = BASELINE_USER_PROMPT.format(
        question=question,
        context=context,
    )

    # Generate the final response
    response = llm.invoke([
        SystemMessage(content=BASELINE_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ])

    return {
        "answer": response.content,
        "docs": docs,
        "context": context,
    }

In [40]:
# Execute the modified baseline_answer function definition
def baseline_answer(question: str, k: int = 4) -> dict:
    """
    Generate an answer using the baseline RAG workflow.

    Steps:
    1. Retrieve relevant policy context
    2. Build the grounded prompt
    3. Send the prompt to the LLM
    4. Return the generated answer and retrieved sources
    """

    # Retrieve relevant chunks from Chroma
    docs = retrieve_docs(question, k=k)

    # Convert retrieved documents into prompt-ready context
    context = format_docs(docs)

    # Inject the runtime values into the prompt template
    user_prompt = BASELINE_USER_PROMPT.format(
        question=question,
        context=context,
    )

    # Generate the final response
    response = llm.invoke([
        SystemMessage(content=BASELINE_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ])

    return {
        "answer": response.content,
        "docs": docs,
        "context": context,
    }


In [42]:
import json

# Run one sample query to verify the end-to-end RAG workflow.
sample_question = (
    "What documents are required for a reimbursement claim, and how does the cashless claim process work?"
)

sample_result = baseline_answer(sample_question)

print(sample_result["answer"])

For a reimbursement claim, the following documents are typically required:

1. A duly completed claim form.
2. Original policy documents.
3. Original receipts from the hospital.
4. Medical reports and discharge summary.
5. Any additional documents requested by the insurer.

The cashless claim process works as follows:

1. Locate a network hospital near you.
2. At the hospital, visit the third-party administrator (TPA) desk and complete the pre-authorisation request form.
3. The hospital's billing team will forward the necessary documents to the insurer for pre-authorisation approval.
4. If approved, the insurer will settle the bill directly with the hospital at the time of discharge.
5. If the cashless request is rejected, you can still proceed with treatment and later apply for a reimbursement claim. 

You can track the claim status through the insurer's mobile app or website.


In [41]:
# Re-run one sample query to verify the end-to-end RAG workflow.
sample_question = (
    "What documents are required for a reimbursement claim, and how does the cashless claim process work?"
)

sample_result = baseline_answer(sample_question)

print("\n--- Generated Answer ---")
print(sample_result["answer"])

print("\n--- Retrieved Context ---")
print(sample_result["context"])

print("\n--- Retrieved Documents (Metadata Only) ---")
for i, doc in enumerate(sample_result["docs"]):
    print(f"Document {i+1} Metadata: {doc.metadata}")



--- Generated Answer ---
For a reimbursement claim, the following documents are typically required:

1. A duly completed claim form.
2. Original policy documents.
3. Original receipts from the hospital.
4. Medical reports and discharge summary.
5. Any additional documents requested by the insurer.

The cashless claim process works as follows:

1. **Locate a Network Hospital**: Find a hospital that is part of the insurer's network.
2. **Pre-Authorization Request**: At the hospital, visit the TPA desk and complete the pre-authorization request form. You will need to present the patient's health ID card or e-health card, the pre-authorization request form, and a valid government-issued photo ID.
3. **Approval Process**: The hospital's billing team will send the documents to the insurer for approval. The insurer will keep you updated on the claim status, and you can also track it through the insurer's mobile app.
4. **Settlement**: If approved, the insurer will settle the bill directly wi

In [49]:
# Aggressively uninstall and reinstall deepeval to resolve potential persistent ModuleNotFoundError.
try:
    print("Attempting to uninstall existing deepeval installations...")
    %pip uninstall -y deepeval
    print("Reinstalling deepeval...")
    %pip install -qU deepeval
    print("Deepeval reinstallation complete.")
except Exception as e:
    print(f"Error during deepeval reinstall: {e}")

Attempting to uninstall existing deepeval installations...
Found existing installation: deepeval 4.0.3
Uninstalling deepeval-4.0.3:
  Successfully uninstalled deepeval-4.0.3
Reinstalling deepeval...
Deepeval reinstallation complete.


In [51]:
# Create a dummy benchmark dataset for demonstration purposes.
# In a real-world scenario, this dataset would be loaded from a file (e.g., CSV, JSON).

from deepeval.dataset import Golden

# Define some example golden test cases
dummy_goldens = [
    Golden(input="What are the eligibility criteria for a family health insurance plan?",
           actual_output="The eligibility criteria for a family health insurance plan include minimum and maximum entry ages, premium renewability conditions, waiting periods for initial coverage, specific ailments, and pre-existing conditions, as well as a grace period for renewal.",
           expected_output="The eligibility criteria for a family health insurance plan are:\n- Minimum Entry Age: 91 days (floater), 18 years (individual).\n- Maximum Entry Age: Lifelong renewability.\n- Premium Payable on Renewal: May change with regulator approval.\n- Waiting Periods: 30 days (initial), 24 months (specific ailments), 36 months (pre-existing conditions).\n- Grace Period: 30 days.",
           retrieval_context=[
               "Eligibility Criteria for Family Health Insurance Cashless family health insurance plans are available to applicants who meet the eligibility conditions defined in the policy terms. The standard rules are: Criterion Condition Minimum Entry Age Individual policy: 18 years. Floater policy: 91 days, provided at least one insured person is aged 18 or above. Maximum Entry Age Lifelong renewability is generally available. Premium Payable on Renewal May change on renewal, subject to prior approval from the relevant insurance regulator. Waiting Periods Initial waiting period: 30 days. Listed specific ailments: 24 months. Pre-existing conditions: 36 months. Grace Period Typically 30 days after expiry to renew without losing accumulated benefits."
           ]),
    Golden(input="What documents are required for a reimbursement claim, and how does the cashless claim process work?",
           actual_output="For a reimbursement claim, you need a completed claim form, original policy documents, hospital receipts, medical reports, and a discharge summary. The cashless process involves finding a network hospital, pre-authorization, and direct settlement with the hospital if approved.",
           expected_output="For a reimbursement claim, required documents include a duly completed claim form, original policy documents, original receipts from the hospital, medical reports and discharge summary, and any additional documents requested by the insurer.\nThe cashless claim process works by locating a network hospital, visiting the TPA desk for pre-authorization, the hospital forwarding documents for approval, and if approved, the insurer settling the bill directly. If rejected, you can still claim reimbursement.",
           retrieval_context=[
               "Documents for Reimbursement Claim: A duly completed claim form, original policy documents, original receipts from the hospital, medical reports and discharge summary. Cashless Claim Process: Locate a network hospital, visit TPA desk, complete pre-authorization form, hospital forwards documents, insurer approves and settles bill directly. If rejected, opt for reimbursement."
           ]),
    Golden(input="Are pre-existing conditions covered under the policy?",
           actual_output="Yes, pre-existing conditions are covered after a waiting period of 36 months.",
           expected_output="Pre-existing conditions are covered after a waiting period of 36 months from the policy inception date.",
           retrieval_context=[
               "Waiting Periods: Initial waiting period: 30 days. Listed specific ailments: 24 months. Pre-existing conditions: 36 months."
           ])
]

# Instantiate the Golden dataset
benchmark_dataset = Golden(goldens=dummy_goldens)

print(f"Benchmark dataset created with {len(benchmark_dataset.goldens)} golden test cases.")

ValidationError: 1 validation error for Golden
input
  Field required [type=missing, input_value={'goldens': [Golden(input..., images_mapping=None)]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

In [46]:
# Create a dummy benchmark dataset for demonstration purposes.
# In a real-world scenario, this dataset would be loaded from a file (e.g., CSV, JSON).

from deepeval.dataset import Golden

# Define some example golden test cases
dummy_goldens = [
    Golden(input="What are the eligibility criteria for a family health insurance plan?",
           actual_output="The eligibility criteria for a family health insurance plan include minimum and maximum entry ages, premium renewability conditions, waiting periods for initial coverage, specific ailments, and pre-existing conditions, as well as a grace period for renewal.",
           expected_output="""The eligibility criteria for a family health insurance plan are:
- Minimum Entry Age: 91 days (floater), 18 years (individual).
- Maximum Entry Age: Lifelong renewability.
- Premium Payable on Renewal: May change with regulator approval.
- Waiting Periods: 30 days (initial), 24 months (specific ailments), 36 months (pre-existing conditions).
- Grace Period: 30 days.""",
           retrieval_context=[
               "Eligibility Criteria for Family Health Insurance Cashless family health insurance plans are available to applicants who meet the eligibility conditions defined in the policy terms. The standard rules are: Criterion Condition Minimum Entry Age Individual policy: 18 years. Floater policy: 91 days, provided at least one insured person is aged 18 or above. Maximum Entry Age Lifelong renewability is generally available. Premium Payable on Renewal May change on renewal, subject to prior approval from the relevant insurance regulator. Waiting Periods Initial waiting period: 30 days. Listed specific ailments: 24 months. Pre-existing conditions: 36 months. Grace Period Typically 30 days after expiry to renew without losing accumulated benefits."
           ]),
    Golden(input="What documents are required for a reimbursement claim, and how does the cashless claim process work?",
           actual_output="For a reimbursement claim, you need a completed claim form, original policy documents, hospital receipts, medical reports, and a discharge summary. The cashless process involves finding a network hospital, pre-authorization, and direct settlement with the hospital if approved.",
           expected_output="""For a reimbursement claim, required documents include a duly completed claim form, original policy documents, original receipts from the hospital, medical reports and discharge summary, and any additional documents requested by the insurer.\nThe cashless claim process works by locating a network hospital, visiting the TPA desk for pre-authorization, the hospital forwarding documents for approval, and if approved, the insurer settling the bill directly. If rejected, you can still claim reimbursement.""",
           retrieval_context=[
               "Documents for Reimbursement Claim: A duly completed claim form, original policy documents, original receipts from the hospital, medical reports and discharge summary. Cashless Claim Process: Locate a network hospital, visit TPA desk, complete pre-authorization form, hospital forwards documents, insurer approves and settles bill directly. If rejected, opt for reimbursement."
           ]),
    Golden(input="Are pre-existing conditions covered under the policy?",
           actual_output="Yes, pre-existing conditions are covered after a waiting period of 36 months.",
           expected_output="Pre-existing conditions are covered after a waiting period of 36 months from the policy inception date.",
           retrieval_context=[
               "Waiting Periods: Initial waiting period: 30 days. Listed specific ailments: 24 months. Pre-existing conditions: 36 months."
           ])
]

# Instantiate the Golden dataset
benchmark_dataset = Golden(goldens=dummy_goldens)

print(f"Benchmark dataset created with {len(benchmark_dataset.goldens)} golden test cases.")

ModuleNotFoundError: No module named 'deepeval'

In [50]:
# Execute the corrected cell to create the dummy benchmark dataset.
# If there are still 'deepeval' import issues, they will be addressed next.

!pip install -qU deepeval
from deepeval.dataset import Golden, EvaluationDataset # Import EvaluationDataset

# Define some example golden test cases
dummy_goldens = [
    Golden(input="What are the eligibility criteria for a family health insurance plan?",
           actual_output="The eligibility criteria for a family health insurance plan include minimum and maximum entry ages, premium renewability conditions, waiting periods for initial coverage, specific ailments, and pre-existing conditions, as well as a grace period for renewal.",
           expected_output="""The eligibility criteria for a family health insurance plan are:
- Minimum Entry Age: 91 days (floater), 18 years (individual).
- Maximum Entry Age: Lifelong renewability.
- Premium Payable on Renewal: May change with regulator approval.
- Waiting Periods: 30 days (initial), 24 months (specific ailments), 36 months (pre-existing conditions).
- Grace Period: 30 days.""",
           retrieval_context=[
               "Eligibility Criteria for Family Health Insurance Cashless family health insurance plans are available to applicants who meet the eligibility conditions defined in the policy terms. The standard rules are: Criterion Condition Minimum Entry Age Individual policy: 18 years. Floater policy: 91 days, provided at least one insured person is aged 18 or above. Maximum Entry Age Lifelong renewability is generally available. Premium Payable on Renewal May change on renewal, subject to prior approval from the relevant insurance regulator. Waiting Periods Initial waiting period: 30 days. Listed specific ailments: 24 months. Pre-existing conditions: 36 months. Grace Period Typically 30 days after expiry to renew without losing accumulated benefits."
           ]),
    Golden(input="What documents are required for a reimbursement claim, and how does the cashless claim process work?",
           actual_output="For a reimbursement claim, you need a completed claim form, original policy documents, hospital receipts, medical reports, and a discharge summary. The cashless process involves finding a network hospital, pre-authorization, and direct settlement with the hospital if approved.",
           expected_output="""For a reimbursement claim, required documents include a duly completed claim form, original policy documents, original receipts from the hospital, medical reports and discharge summary, and any additional documents requested by the insurer.\nThe cashless claim process works by locating a network hospital, visiting the TPA desk for pre-authorization, the hospital forwarding documents for approval, and if approved, the insurer settling the bill directly. If rejected, you can still claim reimbursement.""",
           retrieval_context=[
               "Documents for Reimbursement Claim: A duly completed claim form, original policy documents, original receipts from the hospital, medical reports and discharge summary. Cashless Claim Process: Locate a network hospital, visit TPA desk, complete pre-authorization form, hospital forwards documents, insurer approves and settles bill directly. If rejected, opt for reimbursement."
           ]),
    Golden(input="Are pre-existing conditions covered under the policy?",
           actual_output="Yes, pre-existing conditions are covered after a waiting period of 36 months.",
           expected_output="Pre-existing conditions are covered after a waiting period of 36 months from the policy inception date.",
           retrieval_context=[
               "Waiting Periods: Initial waiting period: 30 days. Listed specific ailments: 24 months. Pre-existing conditions: 36 months."
           ])
]

# Instantiate the EvaluationDataset with the list of Golden objects
benchmark_dataset = EvaluationDataset(goldens=dummy_goldens)

print(f"Benchmark dataset created with {len(benchmark_dataset.goldens)} golden test cases.")

Benchmark dataset created with 3 golden test cases.


*Prompt:*

I want to prepare a gold benchmark dataset for evaluating my health insurance RAG system. Help me write code to:

Load a CSV benchmark dataset containing question, answer, context, and supporting sources
Validate that all required columns are present before using the dataset
Convert each row into a structured dictionary format
Create DeepEval Golden objects from the benchmark rows
Split the benchmark into train and test sets
Print the number of examples in each split for verification

In [52]:
def load_gold_benchmark(path: Path) -> list[dict]:
    """
    Load the benchmark dataset created for health insurance policy evaluation.

    Required columns:
    - question
    - answer
    - context
    - supporting_sources
    """
    ensure_file_exists(path)

    df = pd.read_csv(path)

    required_cols = ["question", "answer", "context", "supporting_sources"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(
            f"Missing required benchmark columns: {missing_cols}. "
            f"Available columns: {list(df.columns)}"
        )

    rows = []
    for _, row in df.iterrows():
        rows.append({
            "question": str(row["question"]).strip(),
            "answer": str(row["answer"]).strip(),
            "context": str(row["context"]).strip(),
            "supporting_sources": str(row["supporting_sources"]).strip(),
        })

    return rows

In [56]:
from pathlib import Path
import pandas as pd
import random
from deepeval.dataset import Golden # Assuming Golden is needed here, based on subsequent code

# Re-define DATA_DIR and BENCHMARK_CSV to ensure they are available in this scope.
# This is necessary if the kernel was restarted or previous defining cells were not run.
DATA_DIR = Path("data")
BENCHMARK_CSV = DATA_DIR / "golden_benchmark_dataset.csv"

# Load the benchmark examples.
gold_rows = load_gold_benchmark(BENCHMARK_CSV)

if len(gold_rows) != 30:
    raise ValueError(f"Expected 30 gold examples, but found {len(gold_rows)}.")


# Build DeepEval Golden objects.
goldens = []
for row in gold_rows:
    goldens.append(
        Golden(
            input=row["question"],
            expected_output=row["answer"],
            context=[row["context"]],
        )
    )


# Split into 24 training examples and 6 test examples.
random.seed(42)
random.shuffle(goldens)

train_goldens = goldens[:24]
test_goldens = goldens[24:]


# Display verification information
print(f"Gold examples: {len(goldens)}")
print(f"Trainset:      {len(train_goldens)}")
print(f"Test set:      {len(test_goldens)}")

print("\nSample golden:")
print(train_goldens[0])

Gold examples: 30
Trainset:      24
Test set:      6

Sample golden:
input='Can the company change my policy benefits or increase the co-payment percentage in the middle of my 3-year term?' actual_output=None expected_output='No, the terms of your contract are fixed for the policy duration. However, the insurer reserves the right to "revise" or "modify" the product design or premium rates at the time of renewal, subject to regulatory approval. If the product is being withdrawn entirely, the company must notify you at least 90 days before your expiry and provide an option to migrate to a similar existing health product with continuity benefits.' context=["[Retrieved Chunk 1] (Source: Health_Insurance_Policy_TandC.pdf, Page: 22)\nNote: Tenure Discount will not apply if the Insured Person has opted for Premium Payment by Instalments. 5.14 Possibility of Revision of Policy Terms Including Premium Rates The Company may revise or modify the terms of the policy, including premium rates. The I

In [57]:
# Load the benchmark examples.
gold_rows = load_gold_benchmark(BENCHMARK_CSV)

if len(gold_rows) != 30:
    raise ValueError(f"Expected 30 gold examples, but found {len(gold_rows)}.")


# Build DeepEval Golden objects.
goldens = []
for row in gold_rows:
    goldens.append(
        Golden(
            input=row["question"],
            expected_output=row["answer"],
            context=[row["context"]],
        )
    )


# Split into 24 training examples and 6 test examples.
random.seed(42)
random.shuffle(goldens)

train_goldens = goldens[:24]
test_goldens = goldens[24:]


# Display verification information
print(f"Gold examples: {len(goldens)}")
print(f"Trainset:      {len(train_goldens)}")
print(f"Test set:      {len(test_goldens)}")

print("\nSample golden:")
print(train_goldens[0])

Gold examples: 30
Trainset:      24
Test set:      6

Sample golden:
input='Can the company change my policy benefits or increase the co-payment percentage in the middle of my 3-year term?' actual_output=None expected_output='No, the terms of your contract are fixed for the policy duration. However, the insurer reserves the right to "revise" or "modify" the product design or premium rates at the time of renewal, subject to regulatory approval. If the product is being withdrawn entirely, the company must notify you at least 90 days before your expiry and provide an option to migrate to a similar existing health product with continuity benefits.' context=["[Retrieved Chunk 1] (Source: Health_Insurance_Policy_TandC.pdf, Page: 22)\nNote: Tenure Discount will not apply if the Insured Person has opted for Premium Payment by Instalments. 5.14 Possibility of Revision of Policy Terms Including Premium Rates The Company may revise or modify the terms of the policy, including premium rates. The I

*Prompt:*

I want to evaluate a health insurance RAG system using benchmark answers and retrieved context. Help me define evaluation metrics to measure:

Relevance between the generated answer and the expected answer
Faithfulness of the generated answer to the retrieved policy context
Completeness of the answer in covering the important policy details
A DeepEval GPT-based evaluation model to score the outputs
A final list of metrics that can be reused across baseline evaluation, prompt optimization, and runtime tuning

In [58]:
# Define the evaluation criteria.

RELEVANCE_CRITERIA = (
    "Evaluate whether the answer is factually aligned with the expected answer. "
    "Reward answers that preserve the key policy, claim, coverage, exclusion, "
    "waiting-period, or eligibility details and penalize missing or incorrect claims."
)

FAITHFULNESS_CRITERIA = (
    "Evaluate whether the answer stays grounded in the retrieved policy evidence. "
    "Reward answers that clearly rely on retrieved brochure, policy wording, exclusions, "
    "or claim guidance. Penalize unsupported or hallucinated policy statements."
)

COMPLETENESS_CRITERIA = (
    "Evaluate whether the answer fully addresses the policy question using the available context. "
    "Reward answers that cover all important policy points, exclusions, claim conditions, "
    "or procedural details. Penalize answers that are incomplete or miss key information."
)

In [60]:
from deepeval.models import GPTModel
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams
import os

# Evaluation model used by DeepEval metrics.
eval_gpt_model = GPTModel(
    model=EVAL_MODEL,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

# 1. Relevance:
# Checks whether the generated answer aligns with the expected benchmark answer.
relevance_metric = GEval(
    name="Relevance",
    criteria=RELEVANCE_CRITERIA,
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    model=eval_gpt_model,
    threshold=0.7,
)

# 2. Faithfulness:
# Checks whether the answer stays grounded in the retrieved context.
faithfulness_metric = GEval(
    name="Faithfulness",
    criteria=FAITHFULNESS_CRITERIA,
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.CONTEXT,
    ],
    model=eval_gpt_model,
    threshold=0.7,
)

# 3. Completeness:
# Checks whether the answer fully covers the important aspects of the question.
completeness_metric = GEval(
    name="Completeness",
    criteria=COMPLETENESS_CRITERIA,
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.CONTEXT,
    ],
    model=eval_gpt_model,
    threshold=0.7,
)

# Final metric list used across:
# - baseline evaluation
# - GEPA optimization
# - RAG configuration tuning
DEEPEVAL_METRICS = [
    relevance_metric,
    faithfulness_metric,
    completeness_metric,
]

1.7 Evaluate the baseline RAG on the golden examples dataset
*Prompt:*

I want to evaluate my baseline health insurance RAG pipeline against a benchmark dataset. Help me write code to:

Create a reusable evaluation function for any answer-generation pipeline
Run the pipeline on each golden example question
Compare the generated answer with the expected answer and retrieved context
Measure relevance, faithfulness, and completeness using DeepEval
Collect metric scores across all examples
Compute average metric values and an overall score
Display the final evaluation summary for the baseline RAG pipeline

In [61]:
def evaluate_pipeline_on_goldens(
    answer_function,
    goldens_subset,
    label: str,
) -> dict:
    """
    Evaluate any RAG pipeline on a subset of benchmark gold examples.

    Parameters:
    - answer_function:
        A function that takes a question and returns:
        {
            "answer": ...,
            "context": ...
        }

    - goldens_subset:
        Train or test goldens

    - label:
        Display label for logs/results
    """

    metric_scores = {
        metric.name: []
        for metric in DEEPEVAL_METRICS
    }

    for idx, golden in enumerate(goldens_subset, start=1):

        print(f"\nEvaluating Example {idx}/{len(goldens_subset)} [{label}]")

        # Generate pipeline output
        result = answer_function(golden.input)

        # Build DeepEval test case
        test_case = LLMTestCase(
            input=golden.input,
            actual_output=result["answer"],
            expected_output=golden.expected_output,
            context=[result["context"]],
        )

        # Run evaluation metrics
        for metric in DEEPEVAL_METRICS:
            try:
                metric.measure(test_case)

                if metric.score is not None:
                    metric_scores[metric.name].append(metric.score)

                print(f"{metric.name}: {metric.score}")

            except Exception as e:
                print(f"{metric.name} failed on this example: {e}")

    # Compute metric means
    metric_means = {
        name: round(sum(scores) / len(scores), 3)
        if scores else 0.0
        for name, scores in metric_scores.items()
    }

    # Compute overall score
    overall_mean = round(
        sum(metric_means.values()) / len(metric_means),
        3
    ) if metric_means else 0.0

    print("\n" + "=" * 80)
    print(f"{label} Metric Means: {metric_means}")
    print(f"{label}: Final Mean DeepEval Score = {overall_mean}")
    print("=" * 80)

    return {
        "label": label,
        "metric_means": metric_means,
        "overall_mean": overall_mean,
    }